# Transpilación de SVM y Scaler a C++ para Microcontrolador (ESP32-S3)

Este notebook se encarga de cargar los binarios exportados en el entrenamiento y exportar las cabeceras C++ directamente al proyecto PlatformIO en `firmware/Classifier/include/`.

In [1]:
import os
import joblib
from micromlgen import port

# Función para buscar y cargar el archivo .env
def load_env_variables():
    from pathlib import Path
    try:
        start_dir = Path(os.getcwd())
    except:
        start_dir = Path(".")
        
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
            
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
        
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
            
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()

✅ Archivo .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


## 1. Transpilar la SVM a C++

In [2]:
models_dir = os.environ["MODELS_ML_PROTO"]
svm_path = os.path.join(models_dir, "svm_model.bin")
classifier_include_dir = os.path.join(os.environ["PROJECT_ROOT"], "firmware", "Classifier", "include")

print(f"📖 Cargando modelo SVM desde: {svm_path} ...")
svm_model = joblib.load(svm_path)

print("⚡ Transpilando modelo SVM a C++ usando micromlgen...")
cpp_code = port(svm_model)

svm_header_path = os.path.join(classifier_include_dir, "svm_model.h")
os.makedirs(classifier_include_dir, exist_ok=True)

with open(svm_header_path, "w", encoding="utf-8") as f:
    f.write("/*\n")
    f.write(" * =============================================================\n")
    f.write(" *  svm_model.h — Modelo SVM autogenerado por micromlgen\n")
    f.write(" * =============================================================\n")
    f.write(" */\n\n")
    f.write("#pragma once\n")
    f.write(cpp_code)

print(f"✅ Cabecera C++ guardada exitosamente en: {svm_header_path}")

📖 Cargando modelo SVM desde: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/models/myotensor_proto/ml/svm_model.bin ...
⚡ Transpilando modelo SVM a C++ usando micromlgen...
✅ Cabecera C++ guardada exitosamente en: /home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/include/svm_model.h


## 2. Exportar los parámetros del StandardScaler

In [3]:
scaler_path = os.path.join(models_dir, "feature_scaler.bin")
print(f"📖 Cargando StandardScaler desde: {scaler_path} ...")
feature_scaler = joblib.load(scaler_path)

means = feature_scaler.mean_
stds = feature_scaler.scale_

means_str = ", ".join(f"{m:.8f}f" for m in means)
stds_str = ", ".join(f"{s:.8f}f" for s in stds)

scaler_header_path = os.path.join(classifier_include_dir, "scaler_params.h")

with open(scaler_header_path, "w", encoding="utf-8") as f:
    f.write("#pragma once\n")
    f.write("/*\n")
    f.write(" * =============================================================\n")
    f.write(" *  scaler_params.h — Parámetros de normalización del StandardScaler\n")
    f.write(" * =============================================================\n")
    f.write(" */\n\n")
    f.write(f"// Medias de las características (MAV, RMS, WL, ZC, SSC, VAR)\n")
    f.write(f"const float feature_means[6] = {{ {means_str} }};\n\n")
    f.write(f"// Desviaciones estándar de las características (MAV, RMS, WL, ZC, SSC, VAR)\n")
    f.write(f"const float feature_stds[6] = {{ {stds_str} }};\n")

print(f"✅ Parámetros del escalador guardados en: {scaler_header_path}")

📖 Cargando StandardScaler desde: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/models/myotensor_proto/ml/feature_scaler.bin ...
✅ Parámetros del escalador guardados en: /home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/include/scaler_params.h
